# Install dependencies and explore data

In [1]:
!pip install -q gensim

In [3]:
import os, json, time, shutil
from collections import Counter

import numpy as np
import pandas as pd
from gensim.models import Word2Vec, KeyedVectors
from gensim.models.word2vec import LineSentence
from scipy.spatial import distance

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

Input data paths are fixed below. All outputs go to `OUT_DIR`:
- `OUT_DIR/embeddings/` — word embeddings (`.vec`), handed over for the next part

In [4]:
DATA = "/Users/ksenia/Documents"
MED_PATH   = os.path.join(DATA, "QUAERO_FrenchMed_traindev.ospl")
PRESS_PATH = os.path.join(DATA, "QUAERO_FrenchPress_traindev.ospl")

for p in [MED_PATH, PRESS_PATH]:
    print(("OK   " if os.path.exists(p) else "MISS "), p)
assert os.path.exists(MED_PATH) and os.path.exists(PRESS_PATH), "Input files not found - check DATA"

CORPORA = {"med": MED_PATH, "press": PRESS_PATH}

OUT_DIR = os.path.join(DATA, "word2vec_output")
EMB_DIR = os.path.join(OUT_DIR, "embeddings")
RESULTS_DIR = os.path.join(OUT_DIR, "results")
os.makedirs(EMB_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Outputs will be saved to:", OUT_DIR)

OK    /Users/ksenia/Documents/QUAERO_FrenchMed_traindev.ospl
OK    /Users/ksenia/Documents/QUAERO_FrenchPress_traindev.ospl
Outputs will be saved to: /Users/ksenia/Documents/word2vec_output


In [5]:
# Show the first 3 raw lines of each corpus 
for name, path in CORPORA.items():
    print(f"{name}: first 3 lines")
    with open(path, encoding="utf-8") as f:
        for _, line in zip(range(3), f):
            print(repr(line[:200]))
    print()

med: first 3 lines
'EMEA / H / C / 551 \n'
'PRIALT \n'
'Qu ’ est ce que Prialt ? \n'

press: first 3 lines
'Patricia Martin , que voici , que voilà ! oh , bonjour Nicolas Stoufflet . \n'
'France Inter , 7 heures . \n'
'le journal , Simon Tivolle . \n'



In [6]:
WORDS = ["patient", "traitement", "maladie", "solution", "jaune"]

# Corpus statistics: sentences, tokens, vocabulary size, hapaxes - words seen only once
counters, stats = {}, []
for name, path in CORPORA.items():
    cnt, n_sent = Counter(), 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            toks = line.split()
            if toks:
                n_sent += 1
                cnt.update(toks)
    counters[name] = cnt
    stats.append({"corpus": name, "sentences": n_sent, "tokens": sum(cnt.values()),
                  "vocabulary": len(cnt),
                  "hapax (freq=1)": sum(1 for c in cnt.values() if c == 1)})

corpus_stats = pd.DataFrame(stats).set_index("corpus")
corpus_stats.to_csv(os.path.join(RESULTS_DIR, "corpus_stats.csv"))
corpus_stats

,sentences,tokens,vocabulary,hapax (freq=1)
corpus,,,,
med,3021,51896,9104,5594
press,38548,1251561,39654,15402


In [7]:
# Frequency of the 5 target words in each corpus
freq = pd.DataFrame({name: [counters[name][w] for w in WORDS] for name in CORPORA}, index=WORDS)
freq.index.name = "word"
freq.to_csv(os.path.join(RESULTS_DIR, "word_frequencies.csv"))
freq

,med,press
word,,
patient,33,11
traitement,251,53
maladie,65,114
solution,67,100
jaune,9,23


In [8]:
print(corpus_stats.to_string())
print(freq.to_string())

        sentences   tokens  vocabulary  hapax (freq=1)
corpus                                                
med          3021    51896        9104            5594
press       38548  1251561       39654           15402
            med  press
word                  
patient      33     11
traitement  251     53
maladie      65    114
solution     67    100
jaune         9     23


The Med corpus is small, and the Press corpus is 24 times bigger. But medical words appear more often in Med (traitement: 251 vs 53). Some words are very rare (jaune: 9 in Med), so their results may not be reliable.

# Training the 4 word2vec models

Key parameter sg: **sg=0 → CBOW** - predict the word from its context, **sg=1 → Skip-gram** - predict the context from the word).

In [9]:
PARAMS = dict(vector_size=100, min_count=1, window=5, epochs=5, workers=4, seed=42)
ALGOS = {"cbow": 0, "skipgram": 1}

models, train_log = {}, []
for corpus, path in CORPORA.items():
    for algo, sg in ALGOS.items():
        key = f"w2v_{algo}_{corpus}"
        print(f"Training {key} ...")
        t0 = time.time()
        model = Word2Vec(sentences=LineSentence(path), sg=sg, **PARAMS)
        secs = round(time.time() - t0, 1)

        model.wv.save_word2vec_format(os.path.join(EMB_DIR, f"{key}.vec"), binary=False)

        models[key] = model
        train_log.append({"model": key, "corpus": corpus, "algo": algo, "sg": sg,
                          "vocab_size": len(model.wv), "corpus_tokens": model.corpus_total_words,
                          "train_seconds": secs})
        print(f"  done: vocab={len(model.wv)}, {secs}s")

train_df = pd.DataFrame(train_log).set_index("model")
train_df.to_csv(os.path.join(RESULTS_DIR, "training_log.csv"))

train_df

Training w2v_cbow_med ...
  done: vocab=9104, 0.2s
Training w2v_skipgram_med ...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  done: vocab=9104, 0.4s
Training w2v_cbow_press ...
  done: vocab=39654, 2.9s
Training w2v_skipgram_press ...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  done: vocab=39654, 7.5s


,corpus,algo,sg,vocab_size,corpus_tokens,train_seconds
model,,,,,,
w2v_cbow_med,med,cbow,0,9104,51896,0.2
w2v_skipgram_med,med,skipgram,1,9104,51896,0.4
w2v_cbow_press,press,cbow,0,39654,1251561,2.9
w2v_skipgram_press,press,skipgram,1,39654,1251561,7.5


In [10]:
# Sanity check: vectors must not contain NaN and must not be all zeros
for key, m in models.items():
    v = m.wv.vectors
    print(f"{key:20s} NaN: {np.isnan(v).any()}   mean |value|: {np.abs(v).mean():.4f}") 

w2v_cbow_med         NaN: False   mean |value|: 0.0195
w2v_skipgram_med     NaN: False   mean |value|: 0.0446
w2v_cbow_press       NaN: False   mean |value|: 0.0505
w2v_skipgram_press   NaN: False   mean |value|: 0.0909


In [11]:
# List the saved embedding files
for fn in sorted(os.listdir(EMB_DIR)):
    print(f"{fn:35s} {os.path.getsize(os.path.join(EMB_DIR, fn)) / 1e6:8.1f} MB")

w2v_cbow_med.vec                        11.7 MB
w2v_cbow_press.vec                      49.6 MB
w2v_skipgram_med.vec                    11.3 MB
w2v_skipgram_press.vec                  47.7 MB


## Semantic similarity 

###  Method 1: gensim most_similar

In [13]:
TOPN = 10
vectors = {key: KeyedVectors.load_word2vec_format(os.path.join(EMB_DIR, f"{key}.vec"), binary=False)
           for key in models}

nn_gensim = {}
for key, kv in vectors.items():
    for w in WORDS:
        if w in kv:
            nn_gensim[(key, w)] = kv.most_similar(w, topn=TOPN)
        else:
            print(f"[OOV] '{w}' not in vocabulary of {key}")
            nn_gensim[(key, w)] = []

### Method 2: scipy.spatial

In [22]:
def most_similar_scipy(kv, word, topn=TOPN):
    target = kv[word].reshape(1, -1)
    sims = 1 - distance.cdist(target, kv.vectors, metric="cosine")[0]
    res = []
    for idx in np.argsort(-sims):
        if kv.index_to_key[idx] != word:
            res.append((kv.index_to_key[idx], float(sims[idx])))
        if len(res) == topn:
            break
    return res

vectors = {key: KeyedVectors.load_word2vec_format(os.path.join(MODELS_DIR, f"{key}.vec"), binary=False)
           for key in models}
nn_scipy = {(key, w): most_similar_scipy(kv, w) if w in kv else []
            for key, kv in vectors.items() for w in WORDS}

# do both methods return the same top-10 lists
check = pd.DataFrame([{"model": k, "word": w,
                       "same_top10": [x for x, _ in nn_gensim[(k, w)]] == [x for x, _ in nn_scipy[(k, w)]]}
                      for (k, w) in nn_gensim])
check.to_csv(os.path.join(RESULTS_DIR, "gensim_vs_scipy_check.csv"), index=False)
print("gensim == scipy in all cases:", check["same_top10"].all())

gensim == scipy in all cases: True


### Neighbour tables per word 

In [25]:
ORDER = ["w2v_cbow_med", "w2v_skipgram_med", "w2v_cbow_press", "w2v_skipgram_press"]

long_rows = []
for key in ORDER:
    corpus, algo = key.split("_")[2], key.split("_")[1]
    for w in WORDS:
        for rank, (v, s) in enumerate(nn_gensim[(key, w)], 1):
            long_rows.append({"word": w, "model": key, "corpus": corpus, "algo": algo,
                              "rank": rank, "neighbour": v, "similarity": round(s, 4)})
neighbours_long = pd.DataFrame(long_rows)
neighbours_long.to_csv(os.path.join(RESULTS_DIR, "neighbours_all.csv"), index=False)

def neighbours_table(word):
    cols = {}
    for key in ORDER:
        nn = nn_gensim[(key, word)]
        cols[key] = [f"{v} ({s:.2f})" for v, s in nn] + [""] * (TOPN - len(nn))
    return pd.DataFrame(cols, index=pd.Index(range(1, TOPN + 1), name="rank"))

for w in WORDS:
    print(f"\n {w}: freq med={counters['med'][w]}, press={counters['press'][w]}")
    t = neighbours_table(w)
    t.to_csv(os.path.join(RESULTS_DIR, f"neighbours_{w}.csv"))
    display(t)


 patient: freq med=33, press=11


,w2v_cbow_med,w2v_skipgram_med,w2v_cbow_press,w2v_skipgram_press
rank,,,,
1,sont (1.00),qui (1.00),Biaou (0.89),trou (0.95)
2,le (1.00),recommandé (1.00),déluge (0.88),idéologie (0.95)
3,les (1.00),grossesse (1.00),effort (0.88),marié (0.95)
4,Des (1.00),Tysabri (1.00),illustrer (0.88),complément (0.95)
5,que (1.00),avant (1.00),essai (0.88),identification (0.95)
6,peut (1.00),symptômes (1.00),passeport (0.87),truc (0.95)
7,par (1.00),associé (1.00),commentaire (0.87),livret (0.95)
8,du (1.00),qu (1.00),business (0.87),regret (0.95)
9,des (1.00),nécessaire (1.00),client (0.87),circonstance (0.94)



 traitement: freq med=251, press=53


,w2v_cbow_med,w2v_skipgram_med,w2v_cbow_press,w2v_skipgram_press
rank,,,,
1,dans (1.00),par (0.98),transfert (0.94),alimentaire (0.91)
2,chez (1.00),TYSABRI (0.97),calendrier (0.92),potentiel (0.90)
3,Le (1.00),médicaments (0.97),financement (0.92),gendarme (0.89)
4,des (1.00),Le (0.97),pont (0.91),coût (0.89)
5,sur (1.00),le (0.96),combat (0.91),financement (0.89)
6,que (1.00),maladie (0.96),laboratoire (0.91),financier (0.88)
7,par (1.00),risque (0.96),pacte (0.91),légalité (0.87)
8,les (1.00),autres (0.96),magasin (0.91),électronique (0.87)
9,avec (1.00),association (0.96),effectif (0.91),outil (0.87)



 maladie: freq med=65, press=114


,w2v_cbow_med,w2v_skipgram_med,w2v_cbow_press,w2v_skipgram_press
rank,,,,
1,la (1.00),risque (0.99),puissance (0.90),misère (0.84)
2,du (1.00),cancer (0.99),négociation (0.90),garantie (0.84)
3,dans (1.00),administration (0.99),bataille (0.90),assurance (0.84)
4,de (1.00),association (0.99),polémique (0.90),crainte (0.83)
5,des (1.00),infection (0.99),proximité (0.90),totale (0.83)
6,le (1.00),adulte (0.99),menace (0.90),possession (0.82)
7,en (1.00),à (0.99),procédure (0.90),complicité (0.82)
8,et (1.00),après (0.99),douleur (0.89),lourde (0.82)
9,à (1.00),en (0.99),mondialisation (0.89),dictature (0.82)



 solution: freq med=67, press=100


,w2v_cbow_med,w2v_skipgram_med,w2v_cbow_press,w2v_skipgram_press
rank,,,,
1,perfusion (1.00),flacon (0.98),difficulté (0.91),pacifique (0.90)
2,( (1.00),jour (0.98),règle (0.90),alternative (0.88)
3,pour (1.00),100 (0.98),catastrophe (0.89),morale (0.86)
4,) (1.00),fois (0.98),seule (0.89),préalable (0.86)
5,1 (1.00),300 (0.98),forme (0.89),règle (0.85)
6,flacon (1.00),contient (0.98),couverture (0.88),vraie (0.85)
7,2 (1.00),injectable (0.97),légitimité (0.88),modifier (0.85)
8,", (1.00)",150 (0.97),puissance (0.88),difficulté (0.85)
9,ou (1.00),mg (0.97),minorité (0.87),crédibilité (0.85)



 jaune: freq med=9, press=23


,w2v_cbow_med,w2v_skipgram_med,w2v_cbow_press,w2v_skipgram_press
rank,,,,
1,) (1.00),14 (1.00),maillot (0.95),maillot (0.94)
2,( (1.00),24 (1.00),dénommé (0.95),emparé (0.91)
3,. (1.00),23 (1.00),cavalier (0.94),triomphe (0.90)
4,", (1.00)",jours (1.00),Christiane (0.94),Bastad (0.89)
5,: (1.00),18 (1.00),chanteur (0.94),demi-finaliste (0.89)
6,3 (1.00),accroître (1.00),Frankfurter (0.94),Azoulay (0.89)
7,• (1.00),débit (1.00),B (0.94),lauréat (0.89)
8,à (1.00),titane (1.00),égyptien (0.94),sprint (0.89)
9,- (1.00),Fluoxétine (1.00),Samuel (0.94),lion (0.89)


## Comparisons

For each pair of models: number of shared neighbours out of 10 and the Jaccard index `|A∩B| / |A∪B|`.
- (a) same corpus, different approach → effect of the algorithm (CBOW vs Skip-gram)
- (b) same approach, different corpus → effect of the data (type and size: Med vs Press)

In [26]:
def overlap(k1, k2, w):
    a = {x for x, _ in nn_gensim[(k1, w)]}
    b = {x for x, _ in nn_gensim[(k2, w)]}
    inter = a & b
    jac = round(len(inter) / len(a | b), 2) if a | b else np.nan
    return len(inter), jac, ", ".join(sorted(inter))

PAIRS = {
    "(a) med: cbow vs skipgram":   ("w2v_cbow_med", "w2v_skipgram_med"),
    "(a) press: cbow vs skipgram": ("w2v_cbow_press", "w2v_skipgram_press"),
    "(b) cbow: med vs press":      ("w2v_cbow_med", "w2v_cbow_press"),
    "(b) skipgram: med vs press":  ("w2v_skipgram_med", "w2v_skipgram_press"),
}
rows = []
for w in WORDS:
    for label, (k1, k2) in PAIRS.items():
        n, j, common = overlap(k1, k2, w)
        rows.append({"word": w, "comparison": label, "shared/10": n, "jaccard": j, "shared_words": common})
overlap_df = pd.DataFrame(rows)
overlap_df.to_csv(os.path.join(RESULTS_DIR, "overlap.csv"), index=False)
overlap_df

,word,comparison,shared/10,jaccard,shared_words
0,patient,(a) med: cbow vs skipgram,0,0.00,
1,patient,(a) press: cbow vs skipgram,0,0.00,
2,patient,(b) cbow: med vs press,0,0.00,
3,patient,(b) skipgram: med vs press,0,0.00,
4,traitement,(a) med: cbow vs skipgram,2,0.11,"Le, par"
5,traitement,(a) press: cbow vs skipgram,1,0.05,financement
6,traitement,(b) cbow: med vs press,0,0.00,
7,traitement,(b) skipgram: med vs press,0,0.00,
8,maladie,(a) med: cbow vs skipgram,2,0.11,"en, à"
9,maladie,(a) press: cbow vs skipgram,0,0.00,


In [27]:
#shared neighbours per word and comparison type
overlap_summary = overlap_df.pivot(index="word", columns="comparison", values="shared/10")
overlap_summary.to_csv(os.path.join(RESULTS_DIR, "overlap_summary.csv"))
overlap_summary

comparison,(a) med: cbow vs skipgram,(a) press: cbow vs skipgram,(b) cbow: med vs press,(b) skipgram: med vs press
word,,,,
jaune,0,1,0,0
maladie,2,0,0,0
patient,0,0,0,0
solution,1,3,0,0
traitement,2,1,0,0


In [28]:
# mean cosine similarity of the top-10 neighbours
mean_sim = pd.DataFrame({k: [np.mean([s for _, s in nn_gensim[(k, w)]]) if nn_gensim[(k, w)] else np.nan
                             for w in WORDS] for k in ORDER}, index=pd.Index(WORDS, name="word")).round(3)
mean_sim.to_csv(os.path.join(RESULTS_DIR, "mean_similarity.csv"))
mean_sim

,w2v_cbow_med,w2v_skipgram_med,w2v_cbow_press,w2v_skipgram_press
word,,,,
patient,0.999,0.996,0.878,0.947
traitement,1.000,0.964,0.914,0.883
maladie,1.000,0.989,0.897,0.826
solution,0.999,0.975,0.887,0.860
jaune,0.998,0.998,0.941,0.898


**CBOW vs Skip-gram** 

On the small Med corpus, CBOW gives bad results: the closest words are "le", "des" and punctuation, and all similarities are about 1.00. This means the model did not learn much, maybe because the corpus is too small. Skip-gram gives better results and finds medical words (maladie → cancer, infection). On the big Press corpus, both methods give meaningful words. The two methods have only 0–3 common neighbours out of 10, so the choice of method really matters.

**Med vs Press** 

The two corpora have no common neighbours (0 out of 10 for every word). This is because the same word is used differently in each corpus. For example, solution means a liquid in Med (flacon, injectable), but an answer to a problem in Press (alternative). Jaune in Press is about cycling (maillot jaune). In Med, all similarities are close to 1.00 because the corpus is too small. In Press they are lower (0.83–0.94), so the vectors are more different from each other. Rare words (jaune in Med) give random results.